In [ ]:
import fiftyone as fo
import fiftyone.brain as fob

SOURCE_DATASETS =[
    #"cip",
    "cubic",
    "rod",
    #"spiky",
    #"core_shell"
]
DEST_DATASET = "merged_dataset"
compute_embeddings = True

In [ ]:

if fo.dataset_exists(DEST_DATASET):
    fo.delete_dataset(DEST_DATASET)
    dest = fo.Dataset(DEST_DATASET)
else:
    dest = fo.Dataset(DEST_DATASET)

seen = set() #to track seen filepaths
total_added = 0 #to track total samples added from source dataset into the merged dataset


In [ ]:

for name in SOURCE_DATASETS:
    if not fo.dataset_exists(name):
        print(f"Source dataset '{name}' does not exist. Skipping merge.")
        continue

    src_dataset = fo.load_dataset(name)
    new_samples = []
    for sample in src_dataset:
        if sample.filepath in seen:
            continue
        new_samples.append(sample.copy()) #copy sample to avoid modifying original
        seen.add(sample.filepath)
    if new_samples:
        dest.add_samples(new_samples)
        total_added += len(new_samples)
        print(f"Added {len(new_samples)} samples from '{name}' to '{DEST_DATASET}'.") #log progress


In [ ]:
dest.persistent = True
print(f"Merging complete. Total samples in '{DEST_DATASET}': {dest.count()}, added {total_added} new samples.")
if compute_embeddings:
    fob.compute_visualization(
        dest,
        brain_key="rn50_umap",
        model="resnet50-imagenet-torch",
        method="umap",
        create_index=True,  # Must be false for 3d
        force_recompute=False,
        num_dims=2,
        seed=42,
        min_dist=0.2
    )

In [ ]:
session = fo.launch_app(dest)
session.wait()